# F1 Active-Aero — Nested-LOCO + 3-Seed Stability (Kaggle GPU)

Focused follow-up that closes the two remaining reviewer experiments at the primary
horizon $H{=}10$. It is self-contained: it rebuilds the windows from the four raw circuit
CSVs, so you only need the raw-data dataset attached (the same one used by the main
notebook).

| Section | What it does | Reuses saved work? |
|---|---|---|
| A. Nested-LOCO | Re-runs the deep-model benchmark with hyperparameters selected **only on held-in circuits** (an inner validation circuit per fold), removing the Monza-tuning leakage. | No — must retrain (HP change). |
| B. 3-seed stability | Trains each deep model at 3 seeds and reports mean±std AUC. | **Yes** — if `f1_results_bundle.zip` is attached, seed-42 predictions are loaded instead of retrained. |

## Setup on Kaggle
1. **Add Data** → attach your raw-CSV dataset (`monaco_red_bull.csv`, `monza_red_bull.csv`,
   `silverstone_red_bull.csv`, `suzuka_red_bull.csv`).
2. *(Optional, for Section B reuse)* Also attach a dataset containing the unzipped
   `processed/deep_preds/` from `f1_results_bundle.zip` (seed-42 predictions).
3. Settings → **GPU T4**. Internet on.
4. Run All. Outputs: `nested_loco_results.csv`, `seed_stability_results.csv`, bundled zip.

Runtime: ~1.5–2.5 h on a T4 (Section A dominates).


In [ ]:
# ── Setup ────────────────────────────────────────────────────────────────────────
import os, glob, pickle, random, warnings, time, zipfile
from pathlib import Path
import numpy as np, pandas as pd
warnings.filterwarnings("ignore")
import torch, torch.nn as nn, torch.nn.functional as Fn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, average_precision_score, f1_score
print("PyTorch", torch.__version__, "| CUDA", torch.cuda.is_available())
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(s)

In [ ]:
# ── Config + build windows (H=10 only) ───────────────────────────────────────────
MASS_KG, SPEED_THR, GEAR_THR, W, H = 798.0, 240.0, 6, 50, 10
FEATURE_COLS = ["Speed","RPM","nGear","Throttle","Brake","X","Y","Z",
                "Acceleration","Elevation_Delta","Kinetic_Energy_MJ","Longitudinal_Force_N"]
CIRCUITS = None
OUT = Path("/kaggle/working");

def coerce_brake(s):
    if pd.api.types.is_bool_dtype(s): return s.astype(np.float32)
    if s.dtype==object: return s.map({"True":1.,"False":0.,"true":1.,"false":0.,"1":1.,"0":0.}).fillna(0.).astype(np.float32)
    return pd.to_numeric(s,errors="coerce").fillna(0.).astype(np.float32)

def load_clean(path, circuit):
    df = pd.read_csv(path); df["Circuit"]=circuit
    for c in ["Speed","RPM","nGear","Throttle","X","Y","Z","LapNumber","Time_Elapsed_Sec"]:
        if c in df.columns: df[c]=pd.to_numeric(df[c],errors="coerce")
    df["Brake"]=coerce_brake(df["Brake"]) if "Brake" in df.columns else 0.
    df=df.sort_values(["Driver","LapNumber","Time_Elapsed_Sec"],kind="mergesort").reset_index(drop=True)
    if "Acceleration" not in df.columns: df["Acceleration"]=df.groupby(["Driver","LapNumber"])["Speed"].diff().fillna(0.)
    else: df["Acceleration"]=pd.to_numeric(df["Acceleration"],errors="coerce").fillna(0.)
    if "Elevation_Delta" not in df.columns: df["Elevation_Delta"]=df.groupby(["Driver","LapNumber"])["Z"].diff().fillna(0.)
    else: df["Elevation_Delta"]=pd.to_numeric(df["Elevation_Delta"],errors="coerce").fillna(0.)
    sp,ac=df["Speed"].fillna(0.),df["Acceleration"].fillna(0.)
    df["Kinetic_Energy_MJ"]=0.5*MASS_KG*(sp/3.6)**2/1e6; df["Longitudinal_Force_N"]=MASS_KG*(ac/3.6)
    df["LapKey"]=(circuit+"|"+df["Driver"].astype(str)+"|L"+df["LapNumber"].fillna(-1).astype(int).astype(str))
    return df.dropna(subset=FEATURE_COLS).reset_index(drop=True)

def build_windows(dfs):
    Xs,ys,cs=[],[],[]
    for df in dfs:
        c=df["Circuit"].iloc[0]
        for lk,lap in df.groupby("LapKey",sort=False):
            lap=lap.sort_values("Time_Elapsed_Sec").reset_index(drop=True); L=len(lap)
            if L<W+H: continue
            feat=lap[FEATURE_COLS].to_numpy(np.float32); sp=lap["Speed"].to_numpy(np.float32); gr=lap["nGear"].to_numpy(np.float32)
            for i in range(L-W-H+1):
                j=i+W+H-1; ys.append(int(sp[j]>=SPEED_THR and gr[j]>=GEAR_THR)); Xs.append(feat[i:i+W]); cs.append(c)
    return np.stack(Xs).astype(np.float32), np.array(ys,np.int8), np.array(cs,object)

files={}
for p in glob.glob("/kaggle/input/**/*red_bull.csv", recursive=True):
    files[Path(p).stem.split("_")[0].capitalize()]=p
assert len(files)>=2, f"Need raw circuit CSVs attached; found {files}"
DFS=[load_clean(v,k) for k,v in sorted(files.items())]
X, y, circ = build_windows(DFS)
CIRCUITS = sorted(set(circ.tolist()))
print("windows", X.shape, "circuits", CIRCUITS, "X-Mode%", round(y.mean()*100,1))

In [ ]:
# ── Model defs (identical to main notebook) ──────────────────────────────────────
class FocalLoss(nn.Module):
    def __init__(self, alpha=0.25, gamma=2.0): super().__init__(); self.a=alpha; self.g=gamma
    def forward(self, l, t):
        bce=Fn.binary_cross_entropy_with_logits(l,t.float(),reduction="none")
        pt=torch.exp(-bce); at=t*self.a+(1-t)*(1-self.a); return (at*(1-pt)**self.g*bce).mean()
class CConv(nn.Module):
    def __init__(s,ic,oc,k,d=1): super().__init__(); s.p=(k-1)*d; s.c=nn.Conv1d(ic,oc,k,dilation=d)
    def forward(s,x): return s.c(Fn.pad(x,(s.p,0)))
class CNN1D(nn.Module):
    def __init__(s,W,F,ch=64,dr=0.2):
        super().__init__(); s.net=nn.Sequential(CConv(F,ch,5),nn.BatchNorm1d(ch),nn.ReLU(),nn.Dropout(dr),
            CConv(ch,ch*2,5),nn.BatchNorm1d(ch*2),nn.ReLU(),nn.Dropout(dr),CConv(ch*2,ch*2,5),nn.BatchNorm1d(ch*2),nn.ReLU())
        s.head=nn.Sequential(nn.Linear(ch*2,ch),nn.ReLU(),nn.Dropout(dr+0.1),nn.Linear(ch,1))
    def forward(s,x): return s.head(s.net(x.transpose(1,2)).mean(-1)).squeeze(-1)
class CLSTM(nn.Module):
    def __init__(s,F,h=128,nl=2,dr=0.2):
        super().__init__(); s.r=nn.LSTM(F,h,nl,batch_first=True,dropout=dr if nl>1 else 0.)
        s.head=nn.Sequential(nn.Linear(h,h//2),nn.ReLU(),nn.Dropout(dr+0.1),nn.Linear(h//2,1))
    def forward(s,x): o,_=s.r(x); return s.head(o[:,-1,:]).squeeze(-1)
class CGRU(nn.Module):
    def __init__(s,F,h=128,nl=2,dr=0.2):
        super().__init__(); s.r=nn.GRU(F,h,nl,batch_first=True,dropout=dr if nl>1 else 0.)
        s.head=nn.Sequential(nn.Linear(h,h//2),nn.ReLU(),nn.Dropout(dr+0.1),nn.Linear(h//2,1))
    def forward(s,x): o,_=s.r(x); return s.head(o[:,-1,:]).squeeze(-1)
class TBlock(nn.Module):
    def __init__(s,ic,oc,k,d,dr):
        super().__init__(); s.c1=CConv(ic,oc,k,d); s.c2=CConv(oc,oc,k,d); s.b1=nn.BatchNorm1d(oc); s.b2=nn.BatchNorm1d(oc)
        s.dr=nn.Dropout(dr); s.proj=nn.Conv1d(ic,oc,1) if ic!=oc else None
    def forward(s,x):
        r=x if s.proj is None else s.proj(x); x=s.dr(Fn.relu(s.b1(s.c1(x)))); x=s.dr(Fn.relu(s.b2(s.c2(x)))); return Fn.relu(x+r)
class TCN(nn.Module):
    def __init__(s,F,ch=64,k=5,dr=0.2):
        super().__init__(); ic=F; L=[]
        for d in [1,2,4,8]: L.append(TBlock(ic,ch,k,d,dr)); ic=ch
        s.net=nn.Sequential(*L); s.head=nn.Sequential(nn.Linear(ch,ch//2),nn.ReLU(),nn.Dropout(dr+0.1),nn.Linear(ch//2,1))
    def forward(s,x): return s.head(s.net(x.transpose(1,2))[:,:,-1]).squeeze(-1)
class PE(nn.Module):
    def __init__(s,d,ml=200):
        super().__init__(); pe=torch.zeros(ml,d); pos=torch.arange(ml).unsqueeze(1).float()
        dv=torch.exp(torch.arange(0,d,2).float()*(-np.log(10000.)/d)); pe[:,0::2]=torch.sin(pos*dv); pe[:,1::2]=torch.cos(pos*dv)
        s.register_buffer("pe",pe.unsqueeze(0))
    def forward(s,x): return x+s.pe[:,:x.size(1)]
class CTrans(nn.Module):
    def __init__(s,F,dm=64,nh=4,nl=2,ff=128,dr=0.1):
        super().__init__(); s.proj=nn.Linear(F,dm); s.pe=PE(dm)
        s.enc=nn.TransformerEncoder(nn.TransformerEncoderLayer(dm,nh,ff,dr,batch_first=True),nl)
        s.head=nn.Sequential(nn.LayerNorm(dm),nn.Linear(dm,1))
    @staticmethod
    def _m(n,dev): m=torch.triu(torch.ones(n,n,device=dev),1); return m.masked_fill(m==1,float("-inf"))
    def forward(s,x):
        x=s.pe(s.proj(x)); x=s.enc(x,mask=s._m(x.size(1),x.device),is_causal=True); return s.head(x[:,-1,:]).squeeze(-1)

def build(name,F,hidden=128,dropout=0.2):
    if name=="CNN": return CNN1D(W,F,dr=dropout)
    if name=="LSTM": return CLSTM(F,h=hidden,dr=dropout)
    if name=="GRU": return CGRU(F,h=hidden,dr=dropout)
    if name=="TCN": return TCN(F,dr=dropout)
    if name=="Transformer": return CTrans(F,dm=hidden if hidden<=128 else 64,dr=min(dropout,0.1))
DEEP=["CNN","LSTM","GRU","TCN","Transformer"]

In [ ]:
# ── Training helpers ─────────────────────────────────────────────────────────────
BATCH=512
def make_ds(Xa,ya,scaler):
    N,Wd,Fd=Xa.shape; Xs=scaler.transform(Xa.reshape(N,-1)).reshape(N,Wd,Fd).astype(np.float32)
    return TensorDataset(torch.from_numpy(Xs),torch.from_numpy(ya.astype(np.float32)))
@torch.no_grad()
def evalp(model,loader):
    model.eval(); P,Y=[],[]
    for xb,yb in loader: P.append(torch.sigmoid(model(xb.to(DEVICE))).cpu().numpy()); Y.append(yb.numpy())
    return np.concatenate(P),np.concatenate(Y)
def fit(model,tr,va,crit,max_ep=80,patience=12,seed=42):
    set_seed(seed)
    opt=torch.optim.AdamW(model.parameters(),lr=1e-3,weight_decay=1e-4)
    sch=torch.optim.lr_scheduler.ReduceLROnPlateau(opt,mode="max",factor=0.5,patience=5,min_lr=1e-5)
    trl=DataLoader(tr,BATCH,shuffle=True); val=DataLoader(va,BATCH*2)
    best,bs,bad=0.,None,0
    for ep in range(1,max_ep+1):
        model.train()
        for xb,yb in trl:
            xb,yb=xb.to(DEVICE),yb.to(DEVICE); opt.zero_grad(); loss=crit(model(xb),yb); loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(),1.0); opt.step()
        vp,vt=evalp(model,val); ap=average_precision_score(vt,vp) if 0<vt.sum()<len(vt) else 0.5; sch.step(ap)
        if ap>best+1e-5: best,bs,bad=ap,{k:v.detach().clone() for k,v in model.state_dict().items()},0
        else: bad+=1
        if bad>=patience: break
    if bs: model.load_state_dict(bs)
    return model
def train_eval(name,train_circ,test_circ,hidden,dropout,seed,max_ep=80):
    Fd=X.shape[2]; tr_mask=np.isin(circ,train_circ); te_mask=circ==test_circ
    scaler=StandardScaler().fit(X[tr_mask].reshape(tr_mask.sum(),-1))
    idx=np.where(tr_mask)[0]
    try: itr,iva=train_test_split(idx,test_size=0.1,stratify=y[tr_mask],random_state=seed)
    except ValueError: itr,iva=train_test_split(idx,test_size=0.1,random_state=seed)
    crit=FocalLoss(alpha=1-float(y[itr].mean()),gamma=2.0).to(DEVICE)
    set_seed(seed); model=build(name,Fd,hidden,dropout).to(DEVICE)
    model=fit(model,make_ds(X[itr],y[itr],scaler),make_ds(X[iva],y[iva],scaler),crit,max_ep=max_ep,seed=seed)
    prob,yt=evalp(model,DataLoader(make_ds(X[te_mask],y[te_mask],scaler),BATCH*2))
    return roc_auc_score(yt,prob), (prob,yt)
print("helpers ready")

## Section A — Nested-LOCO (held-in-only hyperparameter selection)

For each outer test circuit, hyperparameters are chosen using only the three held-in
circuits (one of them serves as the inner validation circuit); the final model is then
retrained on all three and evaluated on the untouched test circuit. No test-fold data ever
informs hyperparameter selection, removing the Monza-tuning exposure. A small 2-config grid
keeps runtime tractable; widen `HP_GRID` if you have budget.

In [ ]:
# ── Section A: Nested-LOCO ───────────────────────────────────────────────────────
HP_GRID = [dict(hidden=128,dropout=0.2), dict(hidden=64,dropout=0.3)]  # widen if budget allows
INNER_EPOCHS, FINAL_EPOCHS = 35, 80
rows=[]; t0=time.time()
for test_c in CIRCUITS:
    train_cs=[c for c in CIRCUITS if c!=test_c]
    inner_val=train_cs[0]; inner_train=train_cs[1:]          # single inner-validation circuit
    for name in DEEP:
        # inner HP search (train on inner_train (2 circuits), validate on inner_val)
        best_hp,best_auc=None,-1
        for hp in HP_GRID:
            auc,_=train_eval(name,inner_train,inner_val,hp["hidden"],hp["dropout"],seed=42,max_ep=INNER_EPOCHS)
            if auc>best_auc: best_auc,best_hp=auc,hp
        # final: retrain on all 3 train circuits with best HP, eval on test circuit
        test_auc,_=train_eval(name,train_cs,test_c,best_hp["hidden"],best_hp["dropout"],seed=42,max_ep=FINAL_EPOCHS)
        rows.append(dict(model=name,test_circuit=test_c,best_hidden=best_hp["hidden"],
                         best_dropout=best_hp["dropout"],test_auc=round(test_auc,4)))
        print(f"  {name:11s} test={test_c:11s} bestHP={best_hp} -> AUC={test_auc:.4f}")
nl=pd.DataFrame(rows); nl.to_csv(OUT/"nested_loco_results.csv",index=False)
print(f"\nNested-LOCO mean AUC by model ({time.time()-t0:.0f}s):")
print(nl.groupby("model")["test_auc"].agg(["mean","std"]).round(4).to_string())

## Section B — 3-seed stability

Each deep model is trained at three seeds and we report mean±std test AUC per circuit and
overall. If a dataset containing `processed/deep_preds/` (the seed-42 predictions from
`f1_results_bundle.zip`) is attached, seed 42 is **loaded** rather than retrained.

In [ ]:
# ── Section B: 3-seed stability ──────────────────────────────────────────────────
SEEDS=[42,1,2]
# locate optional seed-42 deep_preds to reuse
pred42=None
for d in glob.glob("/kaggle/input/**/deep_preds", recursive=True):
    if any(Path(d).glob("*_H010.npz")): pred42=Path(d); break
print("Reusing seed-42 preds from:", pred42 if pred42 else "none (will train seed 42)")

rows=[]; t0=time.time()
for name in DEEP:
    for seed in SEEDS:
        for test_c in CIRCUITS:
            if seed==42 and pred42 is not None and (pred42/f"{name}_{test_c}_H010.npz").exists():
                d=np.load(pred42/f"{name}_{test_c}_H010.npz",allow_pickle=True)
                auc=roc_auc_score(d["y_true"].astype(int),d["y_prob"].astype(float))
            else:
                train_cs=[c for c in CIRCUITS if c!=test_c]
                auc,_=train_eval(name,train_cs,test_c,128,0.2,seed=seed,max_ep=80)
            rows.append(dict(model=name,seed=seed,test_circuit=test_c,auc=round(auc,4)))
        print(f"  {name:11s} seed={seed} done")
ss=pd.DataFrame(rows); ss.to_csv(OUT/"seed_stability_results.csv",index=False)
# mean across circuits per (model,seed), then mean±std across seeds
per_seed=ss.groupby(["model","seed"])["auc"].mean().reset_index()
summary=per_seed.groupby("model")["auc"].agg(["mean","std"]).round(4)
print(f"\n3-seed stability (mean across circuits, then mean±std across seeds) ({time.time()-t0:.0f}s):")
print(summary.to_string())

In [ ]:
# ── Bundle outputs ───────────────────────────────────────────────────────────────
bundle=OUT/"nested_and_seeds_results.zip"
with zipfile.ZipFile(bundle,"w",zipfile.ZIP_DEFLATED) as z:
    for f in ["nested_loco_results.csv","seed_stability_results.csv"]:
        if (OUT/f).exists(): z.write(OUT/f, f)
print("Saved", bundle)
print("Download nested_loco_results.csv + seed_stability_results.csv from the output panel.")